In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sn
import os
from scipy.signal import medfilt,medfilt2d, find_peaks, resample
from scipy.interpolate import interp1d
from numpy.fft import fft
from scipy.ndimage import gaussian_filter
from tqdm import trange
from scipy.ndimage import zoom
from skimage.measure import label
from skimage.measure import regionprops
import json

from RF_FUNCTIONS import *
from LOADING_FUNCTIONS import *

In [8]:
#Load list of all Experiment IDs used for the RF analysis
with open('exp_names.txt', 'r') as f:
    EXP_NAMES = [line.strip() for line in f if line.strip()] 

gen_path = '/bucket/ReiterU/electrophysiology/squidOL/'
rec_paths = []

for exp_name in EXP_NAMES:
    rec_list = os.listdir(gen_path + exp_name)
    found = False
    for i in rec_list:
        if 'SN_LUM' in i:
            rec_paths.append(f"{gen_path}{exp_name}/{i}")
            found = True
    if not found:
        print(f'No matching recording in {exp_name}!!')
        
#Load dictionary with previously identified neuron IDs with RFs      
with open('response_dict.json', 'r') as f:
    response_dict = json.load(f) 

In [20]:
DEBUG = False # Το control plotting outputs


In [10]:
ALL_STIM_WINDOWS = []
ALL_spikes = []
ALL_ids = []
ALL_channels = []
dataset_ids = []
stims = []
stim_names = []
probe_data = []
ALL_stim_matrices = []
MIX_BINAR = []
COND_VECTORS = []


for rec_index in trange(len(rec_paths)):
    rec_path = rec_paths[rec_index]
    mix_binar = 'SN_LUM_POL_MIX' in rec_path #Flag to deal with Intensity stimuli or Intensity+POL stimuli
    exp_path = os.path.join(gen_path, EXP_NAMES[rec_index])
    post_path = os.path.join(rec_path, 'post_processed')

    for probe in os.listdir(post_path):
        probe_path = os.path.join(post_path, probe)
        if len(os.listdir(probe_path)) != 0:
            MIX_BINAR.append(mix_binar)
            stim_windows = np.load(os.path.join(rec_path, 'STIM_WINDOWS.npy'))
            ALL_STIM_WINDOWS.append(stim_windows)

            sort_path = os.path.join(probe_path)
            dataset_ids.append(f"{EXP_NAMES[rec_index]}_{probe}")

            stim_path, stim_name, cond_vector = load_stim_and_conditions(rec_path, mix_binar)
            stim_matrix = np.load(stim_path)

            if not mix_binar:
                COND_VECTORS.append(np.nan)
            else:
                COND_VECTORS.append(cond_vector)

            stim_names.append(stim_name)
            stims.append(stim_matrix)
            ALL_stim_matrices.append(stim_matrix)

            spike_t_matrix = np.load(os.path.join(sort_path, 'spike_t_matrix.npy'), allow_pickle=True)
            g_channels = np.load(os.path.join(sort_path, 'g_channels.npy'))
            g_ids = np.load(os.path.join(sort_path, 'g_ids.npy'))

            ALL_spikes.append(spike_t_matrix)
            ALL_ids.append(g_ids)
            ALL_channels.append(g_channels)

100%|██████████| 36/36 [00:04<00:00,  7.94it/s]


In [11]:
ALL_stim_FRs = compute_stim_frs(dataset_ids, ALL_STIM_WINDOWS, ALL_ids, ALL_spikes, response_dict) #Bin spikes to compute ETAs

100%|██████████| 55/55 [00:10<00:00,  5.32it/s]


In [12]:
ALL_ON_RESPONSES, ALL_OFF_RESPONSES, ALL_ON_RESPONSES_NULL, ALL_OFF_RESPONSES_NULL = compute_on_off_responses(
dataset_ids, ALL_STIM_WINDOWS, ALL_ids, response_dict,
ALL_spikes, ALL_stim_FRs, ALL_stim_matrices, COND_VECTORS, MIX_BINAR
) #Calculate ETAs for ON sparse noise, OFF sparse noise, and null versions

100%|██████████| 52/52 [01:47<00:00,  2.07s/it]


In [13]:
ALL_ONs, ALL_NONEs, ALL_CENTREs = compute_on_none_responses_and_centers(
dataset_ids, response_dict, ALL_STIM_WINDOWS, ALL_ids,
ALL_spikes, ALL_stim_FRs, ALL_stim_matrices, COND_VECTORS, MIX_BINAR,
ALL_ON_RESPONSES, ALL_OFF_RESPONSES, find_zero_neighborhood_indexes_with_previous
) #Calculate response at RF center during stimulus and without any stimulus to calculate VMI later.

100%|██████████| 52/52 [00:47<00:00,  1.09it/s]


In [ ]:
c = 0
up_factor = 5
PEAK_THRESH = 5
BLOB_THRESH = 5

ALL_SIZES = []
ALL_CENTERS = []
ALL_DIAMS = []
THE_INDEX = 0

for data_index in range(len(dataset_ids)):
    if dataset_ids[data_index] in response_dict:
        print(dataset_ids[data_index])

        ONs = ALL_ONs[c]
        NONEs = ALL_NONEs[c]
        CENTREs = ALL_CENTREs[c]
        c += 1

        stim_windows = ALL_STIM_WINDOWS[data_index]
        g_ids = ALL_ids[data_index]
        g_channels = ALL_channels[data_index]
        response_units = response_dict[dataset_ids[data_index]]
        spike_t_matrix = ALL_spikes[data_index]
        stim_unit_FRs = ALL_stim_FRs[data_index]
        STIM_LENGTH = np.mean(np.diff(stim_windows, axis=1))
        EXTENSION = STIM_LENGTH
        SN_MATRIX = stims[data_index]
        BIN_SIZE = 0.01

        SIZES = []
        CENTERS = []
        DIAMS = []

        for unit in range(len(g_ids)):
            if not np.isnan(ONs[unit]):
                SPIKES = spike_t_matrix[unit]
                ON_MAT = ALL_ON_RESPONSES[data_index][unit]
                OFF_MAT = ALL_OFF_RESPONSES[data_index][unit]
                ON_NULL = ALL_ON_RESPONSES_NULL[data_index][unit]
                OFF_NULL = ALL_OFF_RESPONSES_NULL[data_index][unit]

                ON_HIST, OFF_HIST, time_bins, RF_CENT, is_on = get_rf_PTSH(
                    SN_MATRIX, ON_MAT, OFF_MAT, SPIKES,
                    stim_windows, STIM_LENGTH, EXTENSION, bin_size=BIN_SIZE
                )

                # RF on screen border?
                def is_on_border(center, shape):
                    return any(idx == 0 or idx == dim - 1 for idx, dim in zip(center, shape))
                
                border_flag = is_on_border(RF_CENT, ON_MAT.shape)

                # Plot ON/OFF matrices + histograms
                fig, ax = plt.subplots(2, 2, figsize=(15, 10))

                im_on = ax[0, 0].imshow(ON_MAT, cmap='coolwarm')
                ax[0, 0].scatter(RF_CENT[1], RF_CENT[0], color='black')
                fig.colorbar(im_on, ax=ax[0, 0])

                ax[0, 1].bar(time_bins[:-1], ON_HIST, width=BIN_SIZE, color='black')
                ax[0, 1].vlines([0.1, 0.2], 0, np.max(ON_HIST), linestyle='dashed', color='red')
                for spine in ['top', 'right']:
                    ax[0, 1].spines[spine].set_visible(False)
                    
                im_off = ax[1, 0].imshow(OFF_MAT, cmap='coolwarm')
                ax[1, 0].scatter(RF_CENT[1], RF_CENT[0], color='black')
                fig.colorbar(im_off, ax=ax[1, 0])

                ax[1, 1].bar(time_bins[:-1], OFF_HIST, width=BIN_SIZE, color='black')
                ax[1, 1].vlines([0.1, 0.2], 0, np.max(OFF_HIST), linestyle='dashed', color='red')

                for spine in ['top', 'right']:
                    ax[1, 1].spines[spine].set_visible(False)
                fig.suptitle(
                    f'id: {g_ids[unit]}, chan: {g_channels[unit]}, is_on: {is_on}, '
                    f'resp: {np.round(ONs[unit], 1)}, base {np.round(NONEs[unit], 1)}, '
                    f'vis_mod: {np.round((ONs[unit]-NONEs[unit])/(ONs[unit]+NONEs[unit]), 1)}',
                    fontsize=16
                )
                plt.tight_layout(rect=[0, 0, 1, 0.95])
                if DEBUG:
                    plt.show()
                else:
                    plt.close(fig)

                # RF filtering and blob detection
                ON_BLOB, ON_SIZE, ON_LOC, ON_FILT_STA, ON_RF_DIAMETER, _, _, _ = calc_rf(
                    ON_MAT, ON_NULL, peak_thresh=PEAK_THRESH, blob_thresh=BLOB_THRESH,
                    gaus_f=1, med_f=1, upsample=up_factor, force_polarity=None
                )
                OFF_BLOB, OFF_SIZE, OFF_LOC, OFF_FILT_STA, OFF_RF_DIAMETER, _, _, _ = calc_rf(
                    OFF_MAT, OFF_NULL, peak_thresh=PEAK_THRESH, blob_thresh=BLOB_THRESH,
                    gaus_f=1, med_f=1, upsample=up_factor, force_polarity=None
                )

                ON_LOC = np.round(np.asarray(ON_LOC) / up_factor, 2)

                # Plot blobs and filtered STAs
                fig, ax = plt.subplots(2, 2, figsize=(16, 10))
                fig.suptitle(f'THE_INDEX = {THE_INDEX}, sum = {np.sum(ON_HIST[11:21])}')
                THE_INDEX += 1

                ax[1, 0].imshow(ON_BLOB, cmap='coolwarm')
                ax[1, 0].set_title(f'size {ON_SIZE}, diam {np.round(ON_RF_DIAMETER, 1)}')

                zoom_factor = ON_BLOB.shape[0] / ON_MAT.shape[0]
                for i in range(1, ON_MAT.shape[0]):
                    ax[1, 0].axhline(i * zoom_factor, color='gray', linestyle='--', linewidth=0.5)
                for j in range(1, ON_MAT.shape[1]):
                    ax[1, 0].axvline(j * zoom_factor, color='gray', linestyle='--', linewidth=0.5)

                ax[1, 1].imshow(OFF_BLOB, cmap='coolwarm')
                ax[1, 1].set_title(f'size {OFF_SIZE}, diam {np.round(OFF_RF_DIAMETER, 1)}')

                ax[0, 0].imshow(ON_FILT_STA, cmap='coolwarm')
                ax[0, 0].set_title(f'ON center: {ON_LOC}')
                for i in range(1, ON_MAT.shape[0]):
                    ax[0, 0].axhline(i * zoom_factor, color='gray', linestyle='--', linewidth=0.5)
                for j in range(1, ON_MAT.shape[1]):
                    ax[0, 0].axvline(j * zoom_factor, color='gray', linestyle='--', linewidth=0.5)

                ax[0, 1].imshow(OFF_FILT_STA, cmap='coolwarm')

                for a in ax.ravel():
                    a.axis('off')

                if DEBUG:
                    plt.show()
                else:
                    plt.close(fig)

                # Store results
                if not border_flag:
                    if is_on:
                        SIZES.append(ON_SIZE)
                        CENTERS.append(ON_LOC)
                        DIAMS.append(ON_RF_DIAMETER)
                    else:
                        SIZES.append(OFF_SIZE)
                        CENTERS.append(OFF_LOC)
                        DIAMS.append(OFF_RF_DIAMETER)
                else:
                    SIZES.append(ON_SIZE)
                    CENTERS.append(np.nan)
                    DIAMS.append(np.nan)
            else:
                SIZES.append(np.nan)
                CENTERS.append(np.nan)
                DIAMS.append(np.nan)

        ALL_SIZES.append(SIZES)
        ALL_CENTERS.append(CENTERS)
        ALL_DIAMS.append(DIAMS)

In [ ]:
ALL_ONs=np.asarray(ALL_ONs,dtype='object')
ALL_NONEs=np.asarray(ALL_NONEs,dtype='object')
ALL_SIZES=np.asarray(ALL_SIZES,dtype='object')
ALL_CENTERS=np.asarray(ALL_CENTERS,dtype='object')
ALL_DIAMs=np.asarray(ALL_DIAMS,dtype='object')

In [ ]:
save_path='/home/k/k-tsaridis/SQUID_EPHYS/'
np.save(save_path+'ALL_ONs.npy',ALL_ONs)
np.save(save_path+'ALL_NONEs.npy',ALL_NONEs)
np.save(save_path+'ALL_CENTERS.npy',ALL_CENTERS)
np.save(save_path+'ALL_SIZES.npy',ALL_SIZES)
np.save(save_path+'ALL_DIAMs.npy',ALL_DIAMs)